# C4: does synthetic training data help the stage-1 embedding model (MiniLM)?

Trains MiniLM on your synthetic pairs (with and without hard negatives), on the same number of REAL expert-labelled pairs, and on both, then tests on projects never used for training (LibEST, CM-1, EBT). The test sets are real; synthetic data is only used to train.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Then run the cells from top to bottom. You need the file `synthetic_colab.zip` (from `project/data/` on your Mac).

In [ ]:
# 1. GPU check (should print a Tesla T4 or similar)
!nvidia-smi -L

In [ ]:
# 2. install what Colab does not have
!pip -q install sentence-transformers python-dotenv

In [ ]:
# 3. upload synthetic_colab.zip when the file picker appears, then unpack it
from google.colab import files
import os, zipfile
up = files.upload()
name = next(iter(up))
os.makedirs('/content/c4', exist_ok=True)
zipfile.ZipFile(name).extractall('/content/c4')
os.chdir('/content/c4')
print(os.listdir('.'))
print(len(os.listdir('data/synthetic/user_domains')), 'domain files,', len(os.listdir('data/synthetic/deepseek')), 'deepseek files')

In [ ]:
# 4. settings: use the GPU, take MiniLM from Hugging Face
import os
os.environ['C4_DEVICE'] = 'cuda'
os.environ['EMBEDDING_MODEL_NAME'] = 'all-MiniLM-L6-v2'
os.environ['PYTHONPATH'] = '/content/c4'
import torch; print('cuda available:', torch.cuda.is_available())

In [ ]:
# 5. the experiment (about 30-45 minutes on a T4). If the connection drops, run this cell again: finished runs are kept.
!cd /content/c4 && python -W ignore -u -m agents.traceability_agent.synthetic evaluate --user-dataset data/synthetic/user_domains data/synthetic/deepseek --pairs 2271 --seeds 3 --epochs 2 2>&1 | grep -v -i 'warn'

In [ ]:
# 6. the result table
import json
r = json.load(open('/content/c4/data/results/synthetic_minilm.json'))
print('arm sizes:', r['arm_sizes'])
un = r['arms']['untrained']
print(f"{'arm':48s} emb@5   emb@10  hyb@5   hyb@10")
print(f"{'untrained MiniLM':48s} {un['embeddings']['5']['macro']:.3f}   {un['embeddings']['10']['macro']:.3f}   {un['hybrid']['5']['macro']:.3f}   {un['hybrid']['10']['macro']:.3f}")
for arm, v in r['arms'].items():
    if arm == 'untrained': continue
    m = v['macro_mean']; g = v['gain_vs_untrained']['embeddings']['5']
    print(f"{arm:48s} {m['embeddings']['5']:.3f}   {m['embeddings']['10']:.3f}   {m['hybrid']['5']:.3f}   {m['hybrid']['10']:.3f}   (emb@5 gain {g['mean']:+.3f}, 95% {g['ci95_first_seed']})")

In [ ]:
# 7. download the result file (send this to me)
import shutil
shutil.make_archive('/content/results_synthetic', 'zip', '/content/c4/data/results')
files.download('/content/results_synthetic.zip')